# Matching Logic for SkillMatch

Find the engineers who best match what a project needs, using k-nearest-neighbours (scikit-learn).

## 1. Load the engineers

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

engineers = pd.read_csv("engineers.csv")
engineers.head()

,name,discipline,region,vertical,years_experience,seniority,domain,communication,timezone,bandwidth,availability_status,available_in_weeks
0,Allison Hill,Frontend,EMEA,Media,10,60,26,66,49,80,Available,0
1,Liam Chaudry,Backend,Americas,Logistics,9,47,32,64,85,31,Partially committed,2
2,Pahal Balay,Cloud,Americas,FinTech,13,63,33,63,86,98,Available,0
3,Girik Kamdar,Backend,EMEA,FinTech,22,80,48,38,49,44,Available,0
4,Lance Hoffman,DevOps,APAC,SaaS,11,45,40,45,40,56,Partially committed,3


## 2. The five things I match on

In [2]:
attributes = ["seniority", "domain", "communication", "timezone", "bandwidth"]
engineers[attributes].head()

,seniority,domain,communication,timezone,bandwidth
0,60,26,66,49,80
1,47,32,64,85,31
2,63,33,63,86,98
3,80,48,38,49,44
4,45,40,45,40,56


## 3. Example project needs

The five levels a project would ask for (a senior, client-facing, full-time role).

In [3]:
project = {
    "seniority": 85,
    "domain": 70,
    "communication": 80,
    "timezone": 60,
    "bandwidth": 90,
}

# put the values in the same order as the attributes list
project_vector = np.array([project[a] for a in attributes])
project_vector

array([85, 70, 80, 60, 90])

## 4. Find engineers with k-NN

scikit-learn's NearestNeighbors finds the closest engineers. Euclidean = exact fit.

In [4]:
knn = NearestNeighbors(n_neighbors=10, metric="euclidean")
knn.fit(engineers[attributes].to_numpy())

distances, indices = knn.kneighbors([project_vector])
engineers.iloc[indices[0]][["name", "discipline", "vertical"] + attributes]

,name,discipline,vertical,seniority,domain,communication,timezone,bandwidth
1951,Yahvi Sampath,Backend,Logistics,75,67,77,71,84
1453,Ayush Banerjee,Frontend,SaaS,99,71,91,65,84
2404,Yatan Sawhney,Frontend,Other,83,69,72,76,79
1124,Leela Agrawal,DevOps,Government,83,72,62,70,84
378,Pushti Bal,DevOps,SaaS,94,69,66,46,95
1638,Mitali Modi,Frontend,E-commerce,91,70,64,45,94
629,Chasmum Apte,Backend,EdTech,72,54,90,57,93
622,George Phillips,Frontend,Logistics,88,54,71,74,84
1074,Anusha Pillay,Backend,Other,70,58,73,70,81
1495,Tejas Chowdhury,Data / Database,Media,90,51,66,62,99


## 5. Match %

Turn the distance into an easy 0-100 score (closer = higher).

In [5]:
# the biggest possible distance (all five values 99 apart)
max_distance = np.sqrt(len(attributes) * (99 ** 2))

results = engineers.iloc[indices[0]].copy()
results["match_percent"] = ((1 - distances[0] / max_distance) * 100).round(1)
results[["name", "discipline", "match_percent"]]

,name,discipline,match_percent
1951,Yahvi Sampath,Backend,92.5
1453,Ayush Banerjee,Frontend,91.2
2404,Yatan Sawhney,Frontend,90.5
1124,Leela Agrawal,DevOps,90.2
378,Pushti Bal,DevOps,89.9
1638,Mitali Modi,Frontend,89.6
629,Chasmum Apte,Backend,89.5
622,George Phillips,Frontend,89.1
1074,Anusha Pillay,Backend,88.9
1495,Tejas Chowdhury,Data / Database,88.3


## 6. Potential fit (cosine)

Switch the metric to cosine to match the shape of the needs, not the exact levels. This finds engineers who could stretch into the role.

In [6]:
potential_knn = NearestNeighbors(n_neighbors=10, metric="cosine")
potential_knn.fit(engineers[attributes].to_numpy())

p_dist, p_idx = potential_knn.kneighbors([project_vector])
engineers.iloc[p_idx[0]][["name", "discipline", "vertical"] + attributes]

/Users/jineshnanal/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/jineshnanal/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/jineshnanal/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,name,discipline,vertical,seniority,domain,communication,timezone,bandwidth
293,Ekta Bhalla,DevOps,Telecom,67,55,64,50,67
2921,Gregory Barnes,AI / ML,Other,65,53,62,39,68
1818,Triya Khurana,Networking,Other,46,37,48,32,53
2356,Tanay Chander,Data / Database,E-commerce,43,37,47,33,51
2864,Tara Taylor,Backend,Media,58,51,52,46,66
1411,Ladli Chaudhuri,Networking,Media,61,48,58,50,67
2132,Ekalinga Wali,Data / Database,FinTech,63,52,62,54,68
2817,Vincent Jackson,Networking,E-commerce,57,47,60,41,56
2372,Vamakshi Kala,Backend,Government,53,45,55,46,57
736,Melissa Miller,Networking,Government,55,55,58,41,60


## 7. Adding weights

Make some things matter more. Here seniority and communication count double.

In [7]:
weights = {"seniority": 2, "domain": 1, "communication": 2, "timezone": 1, "bandwidth": 1}
weight_vector = np.array([weights[a] for a in attributes])

weighted_knn = NearestNeighbors(n_neighbors=10, metric="euclidean")
weighted_knn.fit(engineers[attributes].to_numpy() * weight_vector)

w_dist, w_idx = weighted_knn.kneighbors([project_vector * weight_vector])
engineers.iloc[w_idx[0]][["name", "discipline"] + attributes]

,name,discipline,seniority,domain,communication,timezone,bandwidth
1951,Yahvi Sampath,Backend,75,67,77,71,84
2404,Yatan Sawhney,Frontend,83,69,72,76,79
2145,Bhavini Dyal,Frontend,80,51,77,72,76
622,George Phillips,Frontend,88,54,71,74,84
1771,Krish Apte,Backend,84,47,89,58,80
2750,Wyatt Johnson,Other,85,65,77,63,60
641,Joshua Martin,AI / ML,83,77,72,66,65
2640,Jai Kata,Other,86,78,73,83,76
1010,Tracy Wilkerson,Cloud,76,54,76,45,71
2915,Corey Thomas,Other,87,57,72,36,75


## 8. One function for everything

Combines the metric choice and weights. I will use this in the backend next.

In [8]:
def recommend(project, method="euclidean", weights=None, top_n=10):
    if weights is None:
        weights = {a: 1 for a in attributes}
    weight_vector = np.array([weights[a] for a in attributes])

    # apply the weights to both the engineers and the project needs
    engineer_data = engineers[attributes].to_numpy() * weight_vector
    project_point = np.array([project[a] for a in attributes]) * weight_vector

    # find the nearest engineers with k-NN
    knn = NearestNeighbors(n_neighbors=top_n, metric=method)
    knn.fit(engineer_data)
    distances, indices = knn.kneighbors([project_point])

    # turn the distance into an easy 0-100 match score
    if method == "euclidean":
        max_distance = np.sqrt((weight_vector ** 2 * (99 ** 2)).sum())
        match = (1 - distances[0] / max_distance) * 100
    else:  # cosine distance is 1 - similarity
        match = (1 - distances[0]) * 100

    result = engineers.iloc[indices[0]].copy()
    result["match_percent"] = match.round(1)
    return result[["name", "discipline", "vertical"] + attributes + ["match_percent"]]


# test it
recommend(project, method="euclidean", top_n=5)

,name,discipline,vertical,seniority,domain,communication,timezone,bandwidth,match_percent
1951,Yahvi Sampath,Backend,Logistics,75,67,77,71,84,92.5
1453,Ayush Banerjee,Frontend,SaaS,99,71,91,65,84,91.2
2404,Yatan Sawhney,Frontend,Other,83,69,72,76,79,90.5
1124,Leela Agrawal,DevOps,Government,83,72,62,70,84,90.2
378,Pushti Bal,DevOps,SaaS,94,69,66,46,95,89.9


## 9. Quick test

A fixed set of project needs, and the engineer names it returns.

In [9]:
test_project = {
    "seniority": 30,
    "domain": 40,
    "communication": 25,
    "timezone": 50,
    "bandwidth": 80,
}

results = recommend(test_project, method="euclidean", top_n=5)
print(results["name"].tolist())
results

['Raksha Sundaram', 'Quincy Raj', 'Wishi Varma', 'Kashish Lad', 'Oni Issac']


,name,discipline,vertical,seniority,domain,communication,timezone,bandwidth,match_percent
263,Raksha Sundaram,Backend,Telecom,30,45,29,48,77,96.7
2013,Quincy Raj,Frontend,Telecom,31,37,30,58,77,95.3
556,Wishi Varma,Frontend,Telecom,40,41,25,50,83,95.3
2361,Kashish Lad,Networking,Media,23,40,28,50,88,95.0
1908,Oni Issac,Cloud,Telecom,26,48,30,54,82,94.9
